# CLARA · Entrenamiento de detector de balón

Entrena un detector YOLO custom en GPU gratuita de Google Colab.

**Antes de empezar:**
1. Runtime → Change runtime type → T4 GPU
2. Ten lista tu API key de Roboflow
3. Runtime → Run all

In [ ]:
!nvidia-smi

In [ ]:
!pip install ultralytics roboflow --quiet

## Descargar dataset de Roboflow

Reemplaza con tu snippet (de Roboflow → Versions → Download Dataset → YOLOv8 → Show download code)

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="TU_API_KEY")
project = rf.workspace("TU_WORKSPACE").project("clara-balon-voleibol")
dataset = project.version(1).download("yolov8")
print(f'\n✅ Dataset en: {dataset.location}')

## Entrenar

In [ ]:
from ultralytics import YOLO
import os

data_yaml = os.path.join(dataset.location, 'data.yaml')
# Base YOLO11m: mayor recall que el nano en balones chicos/borrosos.
# En T4 entrena bien; baja a 'yolo11s.pt' si te quedas sin VRAM.
model = YOLO('yolo11m.pt')
results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    name='clara_balon',
    patience=20,
    plots=True,
    save=True,
)
print('\n✅ Entrenamiento completado')

## Descargar el modelo

In [ ]:
import shutil, glob
from google.colab import files

best = sorted(glob.glob('runs/detect/*/weights/best.pt'))[-1]
shutil.copy(best, 'clara_balon_v1.pt')
files.download('clara_balon_v1.pt')
print('✅ Descargado clara_balon_v1.pt')